<a href="https://colab.research.google.com/github/Rajapaksha-RMDD/pusl3123-ai-ml-coursework/blob/main/Exercise_2_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN Classifier on CIFAR-10



In [ ]:
import json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "-", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (switch runtime to GPU)")

SEED, EPOCHS, BATCH_SIZE, LR = 42, 10, 64, 1e-3

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

## Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

train_full   = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_dataset, val_dataset = random_split(
    train_full, [45000, 5000], generator=torch.Generator().manual_seed(42)
)

train_loader      = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                               generator=torch.Generator().manual_seed(SEED))
val_loader        = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader       = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
train_eval_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

classes = train_full.classes
print("Train:", len(train_dataset), "| Validation:", len(val_dataset), "| Test:", len(test_dataset))

## Mode CNN with three trainable layers


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)    # trainable layer 1
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)   # trainable layer 2
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 8 * 8, 10)                     # trainable layer 3

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # 32x32 -> 16x16
        x = self.pool(torch.relu(self.conv2(x)))   # 16x16 -> 8x8
        x = torch.flatten(x, 1)
        return self.fc1(x)

set_seed(SEED)
model = CNN().to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", total_params)       # expected 60,362

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

## Training and evaluation functions

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss_sum += nn.functional.cross_entropy(out, y, reduction="sum").item()
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, 100.0 * correct / total

def train_one_epoch(model, loader):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, 100.0 * correct / total

## Train for 10 epochs

In [ ]:
history = {"training_loss": [], "validation_loss": [], "train_accuracy": [], "val_accuracy": []}
t0 = time.time()
for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_one_epoch(model, train_loader)
    va_loss, va_acc = evaluate(model, val_loader)
    for k, v in zip(history, (tr_loss, va_loss, tr_acc, va_acc)):
        history[k].append(v)
    print(f"Epoch [{epoch+1:2d}/{EPOCHS}] train loss {tr_loss:.4f} acc {tr_acc:.2f}% | "
          f"val loss {va_loss:.4f} acc {va_acc:.2f}%")
print(f"Training time: {time.time()-t0:.0f} sec")

In [ ]:
ep = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(ep, history["training_loss"], marker="o", label="Training loss")
ax[0].plot(ep, history["validation_loss"],   marker="o", label="Validation loss")
ax[0].set_title("CNN – training and validation loss"); ax[0].set_ylabel("Loss")
ax[1].plot(ep, history["training_acc"], marker="o", label="Training accuracy")
ax[1].plot(ep, history["validation_acc"],   marker="o", label="Validation accuracy")
ax[1].set_title("CNN – training and validation accuracy"); ax[1].set_ylabel("Accuracy (%)")
for a in ax: a.set_xlabel("Epoch"); a.set_xticks(list(ep)); a.grid(True); a.legend()
plt.tight_layout(); plt.savefig("ex2_cnn_curves.png", dpi=150); plt.show()

## Final evaluation

In [ ]:
train_loss_eval, train_acc_eval = evaluate(model, train_eval_loader)
val_loss_f, val_acc_f   = evaluate(model, val_loader)
test_loss_f, test_acc_f = evaluate(model, test_loader)

print("CNN final results")
print("-" * 40)
print(f"Training   accuracy: {train_acc_eval:.2f}%   loss: {train_loss_eval:.4f}")
print(f"Validation accuracy: {val_acc_f:.2f}%   loss: {val_loss_f:.4f}")
print(f"Test       accuracy: {test_acc_f:.2f}%   loss: {test_loss_f:.4f}")
print(f"Trainable parameters: {total_params:,}")
print(f"Train - test gap    : {train_acc_eval - test_acc_f:.2f} percentage points")

cnn_results = {
    "train_accuracy": train_acc_eval, "validation_accuracy": val_acc_f, "test_accuracy": test_acc_f,
    "train_loss_final_epoch": history["train_loss"][-1], "validation_loss": val_loss_f, "test_loss": test_loss_f,
    "parameters": total_params, "history": history,
}
json.dump(cnn_results, open("cnn_results.json", "w"), indent=2)
torch.save(model.state_dict(), "cnn_cifar10_model.pth")

In [ ]:
from sklearn.metrics import confusion_matrix
model.eval()
preds, trues = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.extend(model(x.to(device)).argmax(1).cpu().numpy()); trues.extend(y.numpy())
preds, trues = np.array(preds), np.array(trues)
for i, c in enumerate(classes):
    print(f"{c:12s}: {100 * (preds[trues == i] == i).mean():.2f}%")

cm = confusion_matrix(trues, preds)
plt.figure(figsize=(8, 7)); plt.imshow(cm); plt.colorbar()
plt.xticks(range(10), classes, rotation=45); plt.yticks(range(10), classes)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("CNN confusion matrix (test set)")
plt.tight_layout(); plt.savefig("ex2_cnn_confusion.png", dpi=150); plt.show()

## What did the convolution layers learn?


In [ ]:
w = model.conv1.weight.detach().cpu()                       # [32, 3, 3, 3]
w = (w - w.min()) / (w.max() - w.min())
fig = plt.figure(figsize=(12, 4))
for i in range(32):
    ax = fig.add_subplot(4, 8, i + 1)
    ax.imshow(w[i].permute(1, 2, 0).numpy(), interpolation="nearest"); ax.axis("off")
plt.suptitle("First-layer convolution filters (3x3 RGB)"); plt.tight_layout(); plt.show()

img, lab = test_dataset[0]
with torch.no_grad():
    fmap = torch.relu(model.conv1(img.unsqueeze(0).to(device))).squeeze(0).cpu()
fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(1, 9, 1); ax.imshow((img * 0.5 + 0.5).permute(1, 2, 0).numpy()); ax.set_title(classes[lab]); ax.axis("off")
for i in range(16):
    ax = fig.add_subplot(2, 9, (i // 8) * 9 + (i % 8) + 2)
    ax.imshow(fmap[i].numpy(), cmap="gray"); ax.axis("off")
plt.suptitle("Original image and 16 of the 32 first-layer feature maps"); plt.tight_layout(); plt.show()

## MLP vs CNN comparison


In [ ]:
import os
if not os.path.exists("mlp_results.json"):
    from google.colab import files
    print("Upload mlp_results.json (from Exercise 1)")
    files.upload()

mlp = json.load(open("mlp_results.json"))

comparison_df = pd.DataFrame({
    "Metric": ["Training Accuracy (%)", "Validation Accuracy (%)", "Test Accuracy (%)",
               "Training Loss (final epoch)", "Validation Loss", "Trainable Parameters"],
    "MLP": [mlp["train_accuracy"], mlp["validation_accuracy"], mlp["test_accuracy"],
            mlp["train_loss_final_epoch"], mlp["validation_loss"], mlp["parameters"]],
    "CNN": [cnn_results["train_accuracy"], cnn_results["validation_accuracy"], cnn_results["test_accuracy"],
            cnn_results["train_loss_final_epoch"], cnn_results["validation_loss"], cnn_results["parameters"]],
})
display(comparison_df.round(4))

print(f"CNN - MLP test accuracy : {cnn_results['test_accuracy'] - mlp['test_accuracy']:+.2f} percentage points")
print(f"MLP gap (train-test)    : {mlp['train_accuracy'] - mlp['test_accuracy']:.2f} points")
print(f"CNN gap (train-test)    : {cnn_results['train_accuracy'] - cnn_results['test_accuracy']:.2f} points")
print(f"MLP has {mlp['parameters'] / cnn_results['parameters']:.1f}x more parameters than the CNN")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ep = range(1, EPOCHS + 1)
ax[0].plot(ep, mlp["history"]["train_loss"], "--o", label="MLP train")
ax[0].plot(ep, mlp["history"]["val_loss"],   "-o",  label="MLP validation")
ax[0].plot(ep, history["train_loss"], "--s", label="CNN train")
ax[0].plot(ep, history["val_loss"],   "-s",  label="CNN validation")
ax[0].set_title("Loss: MLP vs CNN"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].grid(True); ax[0].legend()

names = ["Train", "Validation", "Test"]
m = [mlp["train_accuracy"], mlp["validation_accuracy"], mlp["test_accuracy"]]
c = [cnn_results["train_accuracy"], cnn_results["validation_accuracy"], cnn_results["test_accuracy"]]
x = np.arange(3)
b1 = ax[1].bar(x - 0.2, m, 0.4, label="MLP"); b2 = ax[1].bar(x + 0.2, c, 0.4, label="CNN")
ax[1].bar_label(b1, fmt="%.1f"); ax[1].bar_label(b2, fmt="%.1f")
ax[1].set_xticks(x); ax[1].set_xticklabels(names); ax[1].set_ylabel("Accuracy (%)")
ax[1].set_title("Accuracy: MLP vs CNN"); ax[1].legend(); ax[1].grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig("ex2_mlp_vs_cnn.png", dpi=150); plt.show()